# Lab 1 - Generate and profile raw data

1. Load the **Octank product catalog** (~1,200 products, seed data shipped in the repo) and profile it.
2. **Generate ~1,000 raw shopper requests** with `gpt-oss-120b` on Bedrock - the exact technique
   the workshop team used for the 20,000-row instructor dataset.
3. **Profile** them: completeness, distributions, skew, coverage gaps.

The generator deliberately plants data-quality problems (duplicates, missing fields, toxic and
ambiguous requests, contradictions, skew) so Lab 2 has real issues to find and fix.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("attendee")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

## 1. The catalog

In [ ]:
from octank import catalog
raw_cat = catalog.load_raw()
print(json.dumps(catalog.profile(raw_cat), indent=1, default=str))
raw_cat.sample(5, random_state=1)

The catalog has defects: inconsistent category spellings, duplicate SKUs, items with no interest
tags. Cleaning is deterministic (canonical categories, de-dupe SKUs, back-fill tags from the
description), so everyone gets the same clean catalog.

In [ ]:
cat = catalog.clean(raw_cat)
print(len(raw_cat), "->", len(cat), "items")
print(json.dumps(catalog.profile(cat), indent=1, default=str))

## 2. Generate raw requests

**Technique:** sample a structured *spec* per request (relationship, age, occasion, budget,
interests, things to avoid, a note) from configurable distributions, then have `gpt-oss-120b`
write the free-text message a real shopper would type, in varied voices. About 45% of specs also
get a **situation** ("recently got sober", "tiny studio apartment") that the message mentions in
passing - it implies something to avoid that the form never says. Finally, plant flaws.
Generation is **resumable** - re-running reuses cached messages.

In [ ]:
from octank import generate
from octank.pipeline import ATTENDEE, Run
run = Run(S, ATTENDEE)
spec_example = generate.sample_spec(__import__("random").Random(0), 0)
print("example spec:", json.dumps(spec_example, indent=1))

In [ ]:
%%time
raw = run.generate_raw()          # ~1,000 rows, a few minutes
print(len(raw), "rows")
raw[["request_id", "message", "occasion", "budget_total", "num_items", "interests", "avoid",
     "situation", "hidden_avoid"]].head(8)

## 3. Profile

The hidden `_flaw` column is the **answer key** - we use it only to score curation at the end of
Lab 2. A real dataset doesn't have one, so we profile without looking at it.

In [ ]:
from octank import profile
profile.completeness(raw)

In [ ]:
print("occasion skew:", profile.skew_report(raw, "occasion"))
print("text stats:", profile.text_stats(raw))
profile.plot_overview(raw, "Raw requests");

In [ ]:
gaps = profile.coverage_gaps(raw, min_share=ATTENDEE.gap_min_share)
print(f"{len(gaps)} under-covered segments (target: >= {ATTENDEE.gap_min_share:.0%} of rows each)")
gaps

**What we learned:** ~60% of requests are Christmas or birthdays, several occasions and the
under-$25 budget band barely appear, and some form fields are missing. Next: clean it up.

> **Agent skill:** this step follows the `aws-ai-ml` skill's **dataset-evaluation** workflow. You can ask the
> SageMaker AI agent: *"Profile my raw requests file for completeness and skew"* and it generates the equivalent cells.